<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Python_Example_for_Group_Project_Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import time
import random

# IMPORTANT: The API key is automatically provided in the Canvas environment.
# Do not modify this line.
# To this (replace 'YOUR_API_KEY_HERE' with your actual key)
#API_KEY = "YOUR_API_KEY_HERE"
MODEL_ID = "gemini-2.5-flash-preview-05-20"
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_ID}:generateContent?key={API_KEY}"

def generate_text_with_retry(prompt, retries=5, delay=1):
    """
    Generates text using the Gemini API with exponential backoff for retries.
    """
    for i in range(retries):
        try:
            chat_history = []
            chat_history.append({"role": "user", "parts": [{"text": prompt}]})
            payload = {"contents": chat_history}

            response = requests.post(API_URL, json=payload)
            response.raise_for_status()  # Raise an exception for bad status codes

            result = response.json()
            if result.get("candidates") and len(result["candidates"]) > 0 and \
               result["candidates"][0].get("content") and \
               result["candidates"][0]["content"].get("parts") and \
               len(result["candidates"][0]["content"]["parts"]) > 0:
                text = result["candidates"][0]["content"]["parts"][0]["text"]
                return text
            else:
                print(f"Warning: Unexpected API response structure on attempt {i+1}.")
                print(f"Response content: {json.dumps(result, indent=2)}")
                raise ValueError("Unexpected API response structure.")

        except requests.exceptions.RequestException as e:
            print(f"API call failed on attempt {i+1}: {e}")
            if i < retries - 1:
                sleep_time = delay * (2 ** i) + random.uniform(0, 1)
                print(f"Retrying in {sleep_time:.2f} seconds...")
                time.sleep(sleep_time)
            else:
                print("All retries failed. Giving up.")
                return None
        except ValueError as e:
            print(f"Error parsing API response on attempt {i+1}: {e}")
            return None
    return None

def refine_prompt(base_prompt, language_pair):
    """
    Simulates a team-based prompt refinement process.
    This function adds details to make the prompt more specific and realistic.
    """
    language1, language2 = language_pair
    print(f"\n--- Group is refining the prompt for '{language1}' to '{language2}' ---")

    # Team member A: The "Data Realism" specialist
    refined_prompt = f"Original task: {base_prompt}. "
    refined_prompt += f"Generate a conversational text between two friends. "

    # Team member B: The "Domain Expert"
    refined_prompt += f"The scenario is a casual conversation about planning a vacation. "

    # Team member C: The "Linguistic Specialist"
    refined_prompt += f"Generate the conversation in {language1} and then translate it to {language2}. "
    refined_prompt += f"Ensure the tone is informal and friendly, using common phrases and slang for each language."

    return refined_prompt

def generate_data_loop(refined_prompt, num_examples=3):
    """
    Implements the data generation loop for a refined prompt.
    """
    print(f"\n--- Starting data generation loop ({num_examples} examples) ---")
    generated_data = []
    for i in range(num_examples):
        print(f"Generating example {i+1}...")

        # The final prompt is sent to the LLM to generate the data
        prompt_with_instructions = f"Refined Prompt: {refined_prompt}. Do not provide any additional text or explanation."

        generated_text = generate_text_with_retry(prompt_with_instructions)

        if generated_text:
            generated_data.append(generated_text)
            print("Successfully generated text.")
        else:
            print("Failed to generate text.")

    return generated_data

if __name__ == "__main__":
    # Define a group project scenario
    group_members = ["Alice (Data Realism)", "Bob (Domain Expert)", "Charlie (Linguistic Specialist)"]
    print(f"Project Team: {', '.join(group_members)}")

    # Define the chosen language pair
    language_pair = ("Spanish", "English")
    print(f"Target Language Pair: {language_pair[0]} to {language_pair[1]}")

    # Define the initial, broad prompt
    base_prompt = "Generate a conversation and its translation."

    # The group refines the prompt based on their expertise
    final_prompt = refine_prompt(base_prompt, language_pair)
    print("\n--- Final Refined Prompt ---")
    print(final_prompt)

    # The group implements and runs the data generation loop
    generated_examples = generate_data_loop(final_prompt, num_examples=2)

    # Display the results of the project time
    print("\n\n--- Project Output: Generated Synthetic Data ---")
    for i, example in enumerate(generated_examples):
        print(f"\nExample {i+1}:\n{example}")

Project Team: Alice (Data Realism), Bob (Domain Expert), Charlie (Linguistic Specialist)
Target Language Pair: Spanish to English

--- Group is refining the prompt for 'Spanish' to 'English' ---

--- Final Refined Prompt ---
Original task: Generate a conversation and its translation.. Generate a conversational text between two friends. The scenario is a casual conversation about planning a vacation. Generate the conversation in Spanish and then translate it to English. Ensure the tone is informal and friendly, using common phrases and slang for each language.

--- Starting data generation loop (2 examples) ---
Generating example 1...
Successfully generated text.
Generating example 2...
Successfully generated text.


--- Project Output: Generated Synthetic Data ---

Example 1:
**Spanish Conversation**

**Sofía:** Oye, Ana, ¿qué tal el finde?
**Ana:** ¡Hola, Sofi! Genial, la verdad. Estuve pensando... ¿Te acuerdas que hablamos de hacer un viaje?
**Sofía:** ¡Sí, claro! ¿Qué tienes en ment